In [3]:
import numpy as np 
import random
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA, IncrementalPCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_validate
from sklearn.model_selection import GroupKFold
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import pickle

In [ ]:
train_data=np.load('Data+Description\\fashion_train.npy')  
train_data_X=train_data[:,:-1]
train_data_y=train_data[:,-1]

In [ ]:
with open("pca_model_50_components.pkl", "rb") as file:
    pca_loaded = pickle.load(file)

# Apply the loaded PCA model to new data
X_new_reduced = pca_loaded.transform(train_data_X)  # X_new is new data


In [34]:
class Layer():
    def __init__(self, n_num, input_dim):
        self.weights = [0]*n_num
        self.weights=self.initialize_parameters(input_dim)
        

    def initialize_parameters(self, input_dim):
        """Initialize parameters with He initialization method"""
        
        parameters = {}
        parameters["W"] = np.random.randn(len(self.weights), input_dim) * np.sqrt(2/input_dim)
        parameters["b"] = np.zeros((len(self.weights), 1))

        return parameters



In [ ]:
class FFNN():
    def __init__(self):
        self.layers=[]
        #self.weights = []
    
    def add_layer(self, n_num, input_dim=0, activation="relu"):
        if input_dim==0:
            input_dim=len(self.layers[-1][0].weights['W'])
        new_layer=Layer(n_num, input_dim)
        self.layers.append([new_layer,activation])
    def fit(self, X_train, Y_train, batch_size, epochs=0, learning_rate=0.001):

        self.minibatch_size = batch_size
        temp_weights=self.layers[1:].copy()
        for i in range(epochs):
            for batch_idx in range(0, int(X_train.shape[1] / batch_size)):
                # Mini Batch Samples
                if batch_idx == int(X_train.shape[1] / batch_size):
                    X = X_train[:, batch_idx*batch_size:]
                    Y = Y_train[:, batch_idx*batch_size:]
                else:
                    X = X_train[:, batch_idx*batch_size: (batch_idx+1)*batch_size]
                    Y = Y_train[:, batch_idx*batch_size: (batch_idx+1)*batch_size]

            I = X

            simple_weights=self.feed_forward(I, temp_weights)
            prediction= simple_weights["A"+str(len(temp_weights))]
        temp_weights.insert(0, self.layers[0])
        self.layers=temp_weights



In [60]:
model = FFNN()
model.add_layer(128, input_dim=784, activation='relu') # First hidden layer
model.add_layer(64, activation='relu')  # Second hidden layer
model.add_layer(32, activation='relu')  # Third hidden layer
model.add_layer(5, activation='softmax')  # Output layer for multi-class classification

#print(model.layers[0][0].weights['W0'].shape)
for l in model.layers:
    print(l[0].weights['W'].shape, l[1])
    #print(l[0].weights['W'], l[1])




(128, 784) relu
(64, 128) relu
(32, 64) relu
(5, 32) softmax


In [63]:
l=[[1],[2],[3]]
temp_weights=l[1:].copy()
for i in model.layers:
    print(i[0].weights['W'].shape)
print(temp_weights)
metrics1={}
metrics=['accuracy', 'loss', 'recall']
for m in metrics: metrics1.update({m : []})
print(metrics1)

(128, 784)
(64, 128)
(32, 64)
(5, 32)
[[2], [3]]
{'accuracy': [], 'loss': [], 'recall': []}


In [ ]:
#define neural network layer
class Layer():
    def __init__(self, n_num, input_dim):
        self.weights = [0]*n_num
        self.weights=self.initialize_parameters(input_dim)
        

    def initialize_parameters(self, input_dim):
        """Initialize parameters with He initialization method"""
        
        parameters = {}
        parameters["W"] = np.random.randn(len(self.weights), input_dim) * np.sqrt(2/input_dim)
        parameters["b"] = np.zeros((len(self.weights), 1))

        return parameters
    
class Adam_optimizer():
    def __init__(self, learning_rate, beta1, beta2, epsilon, decay_rate):
        #momentum_w, rms_w = 0, 0
        #momentum_b, rms_b = 0, 0
        self.beta1 = beta1
        self.beta2 = beta2
        self.epsilon = epsilon
        self.learning_rate = learning_rate
        self.decay_rate = decay_rate  # Added decay rate


    def update_parameters(self, parameters, gradients, epoch):#(self, t, l_num, b, dw, db):
        ## Compute decayed learning rate
        
        ## dw, db are from current minibatch
        ## momentum beta 1
        # *** weights *** #
        
        l_num=0

        for l in parameters:

            momentum_w=0
            momentum_b=0
            rms_w=0
            rms_b=0

            momentum_w = self.beta1 * momentum_w + (1 - self.beta1) * gradients["dW"+str(l_num)]
            # *** biases *** #
            momentum_b = self.beta1 * momentum_b + (1 - self.beta1) * gradients["db"+str(l_num)]

            ## rms beta 2
            # *** weights *** #
            rms_w = self.beta2 * rms_w + (1 - self.beta2) * (gradients["dW"+str(l_num)]**2)
            # *** biases *** #
            rms_b = self.beta2 * rms_b + (1 - self.beta2) * (gradients["db"+str(l_num)]**2)

            ## bias correction
        
            ## update weights and biases
            decayed_learning_rate = self.learning_rate * np.exp(-self.decay_rate * epoch)
            
            momentum_w_corr = momentum_w / (1 - self.beta1**epoch)
            momentum_b_corr = momentum_b / (1 - self.beta1**epoch)
            rms_w_corr = rms_w / (1 - self.beta2**epoch)
            rms_b_corr = rms_b / (1 - self.beta2**epoch)
            #l_num = l_num - decayed_learning_rate * (m_dw_corr / (np.sqrt(v_dw_corr) + self.epsilon))
            #b = b - decayed_learning_rate * (m_db_corr / (np.sqrt(v_db_corr) + self.epsilon))
        
            l[0].weights['W'] -= decayed_learning_rate * (momentum_w_corr / (np.sqrt(rms_w_corr) + self.epsilon)) # decayed_learning_rate * gradients["dW"+str(l)]
            l[0].weights['b'] -= decayed_learning_rate * (momentum_b_corr / (np.sqrt(rms_b_corr) + self.epsilon)) #decayed_learning_rate * gradients["db"+str(l)]
            l_num+=1

        return parameters

class Mini_Batch_GD_optimizer():
    def __init__(self, learning_rate, decay_rate=0.01):
        #gradients=self.find_gradients(parameters, forward_vars, Y)
        #self.update_parameters(parameters, gradients, learning_rate)
        self.learning_rate=learning_rate
        self.decay_rate=decay_rate
    
    def update_parameters(self, parameters, gradients, epoch):
        decayed_learning_rate = self.learning_rate * np.exp(-self.decay_rate * epoch)
        l_num=0
        for l in parameters:
            l[0].weights['W'] -= decayed_learning_rate * gradients["dW"+str(l_num)]
            l[0].weights['b'] -= decayed_learning_rate * gradients["db"+str(l_num)]
            l_num+=1
        return parameters
    
#define fnn    
class FFNN():
    def __init__(self):
        self.layers=[]
        self.optimizer=None
        self.metrics={}
        #self.weights = []
    
    def add_layer(self, n_num, input_dim=0, activation="relu"):
        if input_dim==0:
            input_dim=len(self.layers[-1][0].weights['W'])
        new_layer=Layer(n_num, input_dim)
        self.layers.append([new_layer,activation])
    
    def compile(self, optimizer='adam', learning_rate=0.001, beta1=0.9, beta2=0.999, epsilon=1e-8, decay_rate=0.01, metrics=['accuracy']):
        if optimizer=='adam':
            self.optimizer=Adam_optimizer(learning_rate, beta1, beta2, epsilon, decay_rate)
        elif optimizer=='mini':
            self.optimizer=Mini_Batch_GD_optimizer(learning_rate, decay_rate)
        for m in metrics: self.metrics.update({m : []})
        
        
    def fit(self, X_train, Y_train, batch_size, epochs=0):

        self.minibatch_size = batch_size
        temp_weights=self.layers.copy()
        for i in range(epochs):
            for batch_idx in range(0, int(X_train.shape[1] / batch_size)):
                # Mini Batch Samples
                if batch_idx == int(X_train.shape[1] / batch_size):
                    X = X_train[:, batch_idx*batch_size:]
                    Y = Y_train[:, batch_idx*batch_size:]
                else:
                    X = X_train[:, batch_idx*batch_size: (batch_idx+1)*batch_size]
                    Y = Y_train[:, batch_idx*batch_size: (batch_idx+1)*batch_size]

                output_history=self.feed_forward( temp_weights, X)
                prediction= output_history["A"+str(len(temp_weights))]
                cost=self.cost(prediction, Y)
                
                gradients=self.backward_prop(temp_weights, output_history, Y)
                temp_weights=self.optimizer.update_parameters(temp_weights, gradients, epoch=i)

        #temp_weights.insert(0, self.layers[0])
        self.layers=temp_weights

    
    def feed_forward(self, temp_weights, I):
        l_num=0
        forward_vars = {"A0": I}
        for layer in temp_weights:      
                #I = np.dot(I, layer[0]) 
            forward_vars["Z"+str(l_num)] = np.dot(layer[0].weights['W'], forward_vars["A"+str(l_num-1)]) + layer[0].weights['b']
                
            if layer == len(self.layers) - 1: 
                    forward_vars["A"+str(l_num)] = self._activate(forward_vars["Z"+str(l_num)], layer[1]) #output layer 
            else: 
                    forward_vars["A"+str(l_num)] = self._activate(forward_vars["Z"+str(l_num)], layer[1]) #hidden layers 
            l_num+=1        
        return {k: v for k, v in forward_vars.items() if k.startswith("A")}

    def backward_prop(self, parameters, forward_vars, Y):
        gradients = {}
        #L = len(self.layer_dims) - 1
        l_num=0
        for l in parameters:
            m = forward_vars["A"+str(l_num-1)].shape[1]
            if l[1] == 'sigmoid':
                gradients["dA"+str(l_num)] = -np.divide(Y, forward_vars["A"+str(l_num)]) + np.divide((1-Y), (1-forward_vars["A"+str(l_num)]))
                sigmoid_derivative = forward_vars["A"+str(l_num)] * (1 - forward_vars["A"+str(l_num)])
                gradients["dZ"+str(l_num)] = np.multiply(gradients["dA"+str(l_num)], sigmoid_derivative)
            elif l[1] == 'softmax': # Compute derivatives if multi class classification
                gradients["dZ"+str(l_num)] = forward_vars["A"+str(l_num)] - Y # derivatives for the last softmax layer
            elif l[1] == 'relu':
                relu_derivative = forward_vars["A"+str(l_num)] > 0
                gradients["dZ"+str(l_num)] = np.multiply(gradients["dA"+str(l_num)], relu_derivative)
                
            gradients["dW"+str(l_num)] = (1/m) * np.dot(gradients["dZ"+str(l_num)], forward_vars["A"+str(l_num-1)].T)
            gradients["db"+str(l_num)] = (1/m) * np.sum(gradients["dZ"+str(l_num)], axis=1, keepdims=True)
            gradients["dA"+str(l_num-1)] = np.dot(parameters.weights['W'].T, gradients["dZ"+str(l_num)])
            l_num+=1
        return gradients
    
    def predict(self, X):
            """Predict labels for given data X"""
            
            forward_vars = self.feed_forward(self.layers, X)
            L = len(self.layers) - 1

            return forward_vars["A"+str(L)]
    
    def _activate(I, activation):
        if activation=='relu':
            result=np.maximum(0, I)
        elif activation=='softmax':
            T = np.exp(I - np.max(I, axis=0, keepdims=True))
            T_sum = np.sum(T, axis=0, keepdims=True)
            result = np.divide(T, T_sum)
        elif activation=='sigmoid':
            result = 1 / (1 + np.exp(-I))
        else:
            print("Error")
        return result
    
    def cost(self, Y_hat, Y):
        """
        Log Loss is applied
        """
        
        m = Y.shape[1]
        if self.binary_classification: 
            cost_value = (1/m) * np.sum(-(Y*np.log(Y_hat) + (1-Y)*np.log(1-Y_hat)))
        else:
            cost_value = (1/m) * np.sum(-(Y * np.log(Y_hat + 1e-15)))

        return cost_value
    
    

In [ ]:
#ceate fnn
fnn_model=FFNN()
fnn_model.add_layer()
fnn_model.fit()


In [ ]:
def create_multiclass_model(input_dim, num_classes):
    model = FFNN()
    model.add_layer(128, input_dim=input_dim, activation='relu') # First hidden layer
    model.add_layer(64, activation='relu')  # Second hidden layer
    model.add_layer(32, activation='relu')  # Third hidden layer
    model.add_layer(num_classes, activation='softmax')  # Output layer for multi-class classification
  
    # Compile the model
    model.compile(optimizer='adam',learning_rate=0.001,
                 #loss='sparse_categorical_crossentropy',  # or 'categorical_crossentropy' if using one-hot encoding
                 metrics=['accuracy'])
    return model

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
fold_accuracies = []
all_classification_reports = []
all_confusion_matrices = []

for fold, (train_index, val_index) in enumerate(kf.split(train_data_X)):
    print(f"Training fold {fold + 1}")
    
    # Split the data into training and validation sets for the current fold
    X_train, X_val = train_data_X[train_index], train_data_X[val_index]
    y_train, y_val = train_data_y[train_index], train_data_y[val_index]
    
    # Create a new instance of the model
    model = create_multiclass_model(input_dim=train_data_X.shape[1], num_classes=len(np.unique(train_data_y)))
    
    # Train the model
    model.fit(X_train, y_train, epochs=10, batch_size=32)
    
    # Evaluate the model on the validation set
    y_val_pred = np.argmax(model.predict(X_val), axis=1)  # Get the predicted class for each sample

    # Calculate accuracy for the current fold
    accuracy = accuracy_score(y_val, y_val_pred)
    fold_accuracies.append(accuracy)

    # Generate classification report and confusion matrix for detailed metrics
    class_report = classification_report(y_val, y_val_pred, output_dict=True)
    conf_matrix = confusion_matrix(y_val, y_val_pred)
    
    # Store the classification report and confusion matrix for the fold
    all_classification_reports.append(class_report)
    all_confusion_matrices.append(conf_matrix)

    print(f"Fold {fold + 1} Accuracy: {accuracy:.4f}")
    print(f"Classification Report for Fold {fold + 1}:\n", classification_report(y_val, y_val_pred))
    print(f"Confusion Matrix for Fold {fold + 1}:\n", conf_matrix)

# Calculate the average accuracy across all folds
average_accuracy = np.mean(fold_accuracies)
print(f"\nAverage Accuracy across 5 folds: {average_accuracy:.4f}")

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
fold_accuracies = []
all_classification_reports = []
all_confusion_matrices = []

for fold, (train_index, val_index) in enumerate(kf.split(X_new_reduced)):
    print(f"Training fold {fold + 1}")
    
    # Split the data into training and validation sets for the current fold
    X_train, X_val = X_new_reduced[train_index], X_new_reduced[val_index]
    y_train, y_val = train_data_y[train_index], train_data_y[val_index]
    
    # Create a new instance of the model
    model = create_multiclass_model(input_dim=X_new_reduced.shape[1], num_classes=len(np.unique(train_data_y)))
    
    # Train the model
    model.fit(X_train, y_train, epochs=10, batch_size=32, verbose=0)
    
    # Evaluate the model on the validation set
    y_val_pred = np.argmax(model.predict(X_val), axis=1)  # Get the predicted class for each sample

    # Calculate accuracy for the current fold
    accuracy_pca = accuracy_score(y_val, y_val_pred)
    fold_accuracies.append(accuracy_pca)

    # Generate classification report and confusion matrix for detailed metrics
    class_report = classification_report(y_val, y_val_pred, output_dict=True)
    conf_matrix = confusion_matrix(y_val, y_val_pred)
    
    # Store the classification report and confusion matrix for the fold
    all_classification_reports.append(class_report)
    all_confusion_matrices.append(conf_matrix)

    print(f"Fold {fold + 1} Accuracy: {accuracy:.4f}")
    print(f"Classification Report for Fold {fold + 1}:\n", classification_report(y_val, y_val_pred))
    print(f"Confusion Matrix for Fold {fold + 1}:\n", conf_matrix)

# Calculate the average accuracy across all folds
average_accuracy_pca = np.mean(fold_accuracies)
print(f"\nAverage Accuracy across 5 folds: {average_accuracy:.4f}")